## 0. Configurations

In [ ]:
from pathlib import Path
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/BSC/Not_in_Text_Explanations')
PROJECT_ROOT = Path('/content/drive/MyDrive/BSC/Not_in_Text_Explanations')

PATHS = {
    'PROJECT_ROOT': PROJECT_ROOT,
    'data_root': PROJECT_ROOT/ 'data',
    'plots': PROJECT_ROOT / 'plots',
    'variables': PROJECT_ROOT / 'variables',
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 1. Import

In [4]:
!pip install captum

In [30]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import seaborn as sns
from scipy.stats import gaussian_kde
from scipy.special import softmax
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import shap
from lime.lime_text import LimeTextExplainer
import torch
from captum.attr import IntegratedGradients
import sentence_generator

# ── Configuration ─────────────────────────────────────────────────────────────
MAX_LENGTH = 512
MAX_EVALS = 100
np.random.seed(42)
# DATASETS =
MODELS = ["distilbert-base-uncased-finetuned-sst-2-english", "siebert/sentiment-roberta-large-english"]
METHODS = ['Occlusion', 'Shap', 'LIME', 'IG']

# ── Style ─────────────────────────────────────────────────────────────────────

sns.set(style='darkgrid')
COLORS = ['#edae49','#d1495b','#00798c', '#30638e']
np.set_printoptions(suppress=True)


In [23]:
from transformers import pipeline
!pip install tqdm
from tqdm.notebook import tqdm

# 2. Utils

In [6]:
def predict_fn(tokenizer, model, texts):
    """
    Given the tokenizer, the model and a batch of texts, obtain the logits using the model.
    """
    inputs = tokenizer(
        [str(t) for t in texts],
        return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=1).cpu().numpy()


def ig_explanation(tokenizer, model, text, max_length=MAX_LENGTH, n_steps=MAX_EVALS):
    """
    Returns token-level attributions using Integrated Gradients
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(model.device)

    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    embeddings = model.get_input_embeddings()(input_ids)
    # I suggest changing these two lines to load a different baseline embedding according to Section 5
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    baseline_emb = model.get_input_embeddings()(baseline_ids)

    def forward_fn(embeds, attention_mask):
        outputs = model(inputs_embeds=embeds, attention_mask=attention_mask)
        return outputs.logits

    with torch.no_grad():
        logits = model(**inputs).logits
        target = torch.argmax(logits, dim=1)
    
    ig = IntegratedGradients(forward_fn)

    attributions = ig.attribute(
        embeddings,
        baselines=baseline_emb,
        additional_forward_args=(attention_mask,),
        target=target,
        n_steps=n_steps
    )

    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = attributions.detach().cpu().numpy()
    attributions = np.abs(attributions)/sum(np.abs(attributions))

    return attributions[1:-1] # Skip first and last token

explanation_fns = {
    'IG': ig_explanation
}

## 3. Loading Pretrained Models

In [7]:
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [8]:
model_name = MODELS[0]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
rob_name = MODELS[1]
rob_tokenizer = AutoTokenizer.from_pretrained(rob_name)
rob_model = AutoModelForSequenceClassification.from_pretrained(rob_name).to(device)
rob_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [9]:
classifier = pipeline("sentiment-analysis", model=MODELS[0], device=3)
rob_classifier = pipeline("sentiment-analysis", model=MODELS[1], device=3)

Device set to use cuda:3
Device set to use cuda:3


# 4. Implementing XAI Methods

## 4.1. Searching a neutral output (Distilbert)

In [11]:
# Aux function for this experiment
def forward_fn_custom(embeds):
    outputs = model(inputs_embeds=embeds)
    return torch.softmax(outputs.logits, dim=1)

In [433]:
# Current baseline I/O
pad_emb = model.get_input_embeddings().weight[tokenizer.pad_token_id]
pad_emb = pad_emb.unsqueeze(0).unsqueeze(1)

print(pad_emb.shape, "-", forward_fn_custom(pad_emb)) # bias towards second class

# Replicating sequence len of the original example
pad_seq = pad_emb.repeat(1, 7, 1)
print(pad_seq.shape, "-", forward_fn_custom(pad_seq)) # this shows bias towards the first class for longer sequences

torch.Size([1, 1, 768]) - tensor([[0.0957, 0.9043]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 768]) - tensor([[0.8670, 0.1330]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [396]:
# Current baseline for pipeline
Baseline = "" + tokenizer.pad_token * 7

bl_tokens = tokenizer(Baseline, return_tensors="pt").to(device)

# Checks if pipeline returns the same results as manual processing
print("Sequence:\n\t", Baseline, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Baseline), "==", torch.softmax(model(**bl_tokens).logits, dim=1))
print()

bl_tokens = {
    "input_ids": bl_tokens.input_ids[:, 1:-1],
    "attention_mask": bl_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**bl_tokens).logits, dim=1))

Sequence:
	 [PAD][PAD][PAD][PAD][PAD][PAD][PAD] 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.8425151109695435}] == tensor([[0.8425, 0.1575]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.8670, 0.1330]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.1.1. Trying different textual inputs

#### Mask

In [399]:
Neutral_example = "" + tokenizer.mask_token * 7 # Close to 0.5

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 [MASK][MASK][MASK][MASK][MASK][MASK][MASK] 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.7235593795776367}] == tensor([[0.7236, 0.2764]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.7661, 0.2339]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Sep

In [401]:
Neutral_example = "" + tokenizer.sep_token * 7 # Close to 0.5

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 [SEP][SEP][SEP][SEP][SEP][SEP][SEP] 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.994438886642456}] == tensor([[0.0056, 0.9944]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.0196, 0.9804]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Cls

In [402]:
Neutral_example = "" + tokenizer.cls_token * 7 # Close to 0.5

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 [CLS][CLS][CLS][CLS][CLS][CLS][CLS] 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.928670346736908}] == tensor([[0.0713, 0.9287]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.4084, 0.5916]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Unk

In [403]:
Neutral_example = "" + tokenizer.unk_token * 7 # Close to 0.5

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 [UNK][UNK][UNK][UNK][UNK][UNK][UNK] 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9200406074523926}] == tensor([[0.9200, 0.0800]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9747, 0.0253]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Average

In [409]:
Neutral_example = "average" + " average" * 6 # Close to 0.5

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 average average average average average average average 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.545901894569397}] == tensor([[0.4541, 0.5459]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9214, 0.0786]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Dots

In [410]:
Neutral_example = "." + " ." * 6

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 . . . . . . . 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.6948005557060242}] == tensor([[0.6948, 0.3052]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9963, 0.0037]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [411]:
Neutral_example = "." * 7

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 ....... 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.6948005557060242}] == tensor([[0.6948, 0.3052]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9963, 0.0037]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Hyphen

In [412]:
Neutral_example = "-" + " -" * 6

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 - - - - - - - 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.6892377138137817}] == tensor([[0.3108, 0.6892]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9960, 0.0040]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [413]:
Neutral_example = "-" * 7

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 ------- 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.6892377138137817}] == tensor([[0.3108, 0.6892]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9960, 0.0040]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Numbers

In [414]:
Neutral_example = "0" * 7

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 0000000 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.9704661965370178}] == tensor([[0.0295, 0.9705]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.8570, 0.1430]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [415]:
Neutral_example = "0" + " 0" * 6

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 0 0 0 0 0 0 0 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.924949049949646}] == tensor([[0.9249, 0.0751]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.0334, 0.9666]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [421]:
from random import randint

Neutral_example = "".join([str(randint(0, 9)) for n in range(7)])

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 0677108 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.5652688145637512}] == tensor([[0.5653, 0.4347]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.7815, 0.2185]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [422]:
from random import randint

Neutral_example = " ".join([str(randint(0, 9)) for n in range(7)])

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

Sequence:
	 1 1 5 1 2 3 1 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'POSITIVE', 'score': 0.9742434024810791}] == tensor([[0.0258, 0.9742]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.0270, 0.9730]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Lorem ipsum

In [424]:
Neutral_example = "Lorem ipsum dolor sit amet consectetur adipiscing"

neutral_tokens = tokenizer(Neutral_example, return_tensors="pt").to(device)
print(neutral_tokens.input_ids.shape)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first [CLS] and last [SEP] tokens:\n\t", classifier(Neutral_example), "==", torch.softmax(model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first [CLS] and last [SEP] tokens:\n\t", torch.softmax(model(**neutral_tokens).logits, dim=1))

torch.Size([1, 19])
Sequence:
	 Lorem ipsum dolor sit amet consectetur adipiscing 

Results with first [CLS] and last [SEP] tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9561273455619812}] == tensor([[0.9561, 0.0439]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first [CLS] and last [SEP] tokens:
	 tensor([[0.9904, 0.0096]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.1.2. Creating a mean embedding

In [280]:
# Trying with a mean embedding...
emb_layer = model.get_input_embeddings().weight # torch.Size([30522, 768])
mean_emb = emb_layer.mean(dim=0, keepdim=True)  # torch.Size([1, 768])
mean_emb = mean_emb.unsqueeze(1)                # torch.Size([1, 1, 768])

print(forward_fn_custom(mean_emb))

tensor([[0.2623, 0.7377]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [303]:
# Trying with a longer sequence...
for n in range(1, 10 + 1):
    mean_seq = mean_emb.repeat(1, n, 1) 
    print(mean_seq.shape, "-", forward_fn_custom(mean_seq)) # this shows bias towards first class for longer sequences and this scenario

torch.Size([1, 1, 768]) - tensor([[0.2623, 0.7377]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 768]) - tensor([[0.9095, 0.0905]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 768]) - tensor([[0.8689, 0.1311]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 768]) - tensor([[0.9470, 0.0530]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 768]) - tensor([[0.9589, 0.0411]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 768]) - tensor([[0.9628, 0.0372]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 768]) - tensor([[0.9596, 0.0404]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 768]) - tensor([[0.9611, 0.0389]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 768]) - tensor([[0.9559, 0.0441]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 768]) - tensor([[0.9586, 0.0414]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [279]:
# Removing unused tokens...
special_tokens_idx = tokenizer.all_special_ids
unused_tokens_idx = [n for n in range(999) if n not in special_tokens_idx]

used_mask = torch.ones(emb_layer.size(0), dtype=torch.bool)
used_mask[unused_tokens_idx] = False

mean_used_emb = emb_layer[used_mask].mean(dim=0, keepdim=True)
mean_used_emb = mean_used_emb.unsqueeze(1)

# results
print(mean_used_emb.shape)
print(forward_fn_custom(mean_used_emb)) # more biased towards second class when compared to the first experiment

torch.Size([1, 1, 768])
tensor([[0.2599, 0.7401]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [304]:
# Trying last experiment with a longer sequence...
for n in range(1, 10 + 1):
    mean_used_seq = mean_used_emb.repeat(1, n, 1) 
    print(mean_used_seq.shape, "-", forward_fn_custom(mean_used_seq)) # this shows bias towards first class for longer sequences again

torch.Size([1, 1, 768]) - tensor([[0.2599, 0.7401]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 768]) - tensor([[0.9142, 0.0858]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 768]) - tensor([[0.8246, 0.1754]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 768]) - tensor([[0.9451, 0.0549]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 768]) - tensor([[0.9591, 0.0409]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 768]) - tensor([[0.9639, 0.0361]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 768]) - tensor([[0.9626, 0.0374]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 768]) - tensor([[0.9650, 0.0350]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 768]) - tensor([[0.9599, 0.0401]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 768]) - tensor([[0.9621, 0.0379]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.1.3. Trying with 0 embedding

In [472]:
zero_emb = torch.zeros((1,1,768)).to(device)

# results
print(zero_emb.shape)
print(forward_fn_custom(zero_emb)) # more bias towards second class when compared to the first experiment

torch.Size([1, 1, 768])
tensor([[0.3788, 0.6212]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [475]:
# Trying last experiment with a longer sequence...
for n in range(1, 10 + 1):
    zero_seq = torch.zeros((1, n, 1)).to(device)
    print(zero_seq.shape, "-", forward_fn_custom(zero_seq)) # this shows bias towards first class for longer sequences again

torch.Size([1, 1, 1]) - tensor([[0.3788, 0.6212]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 1]) - tensor([[0.8647, 0.1353]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 1]) - tensor([[0.7544, 0.2456]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 1]) - tensor([[0.8682, 0.1318]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 1]) - tensor([[0.9735, 0.0265]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 1]) - tensor([[0.9924, 0.0076]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 1]) - tensor([[0.9952, 0.0048]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 1]) - tensor([[0.9954, 0.0046]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 1]) - tensor([[0.9954, 0.0046]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 1]) - tensor([[0.9946, 0.0054]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.1.4. Trying with random embeddings

In [656]:
rand_emb = torch.randn((1, 7, 768)).to(device)

# results
print(rand_emb.shape)
print(forward_fn_custom(rand_emb)) # more biased towards second class when compared to the first experiment

torch.Size([1, 7, 768])
tensor([[0.9651, 0.0349]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [695]:
for n in range(10000):
    for m in range(1, 10+1):
        rand_emb = torch.randn((1, m, 768)).to(device)

        results = forward_fn_custom(rand_emb)
        v1 = results[0, 0].item()
        v2 = results[0, 1].item()
        
        if (0.48 <= v1 <= 0.52):
            folder = Path(f"./ig_baselines/random_embs/distilbert/1_{m}_768/")
            filename = f"{str(v1).replace(".", "")[:5]}_{str(v2).replace(".", "")[:5]}.pt"
            filename =  folder / Path(filename)
            filename.parent.mkdir(parents=True, exist_ok=True)
            torch.save(rand_emb, filename)
            print(filename, results)

./ig_baselines/distilbert/1_3_768/04909_05090.pt tensor([[0.4910, 0.5090]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_2_768/05018_04981.pt tensor([[0.5019, 0.4981]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_7_768/04911_05088.pt tensor([[0.4912, 0.5088]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_2_768/05168_04831.pt tensor([[0.5169, 0.4831]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_4_768/04860_05139.pt tensor([[0.4860, 0.5140]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_2_768/05105_04894.pt tensor([[0.5106, 0.4894]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_10_768/05036_04963.pt tensor([[0.5036, 0.4964]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_3_768/04934_05065.pt tensor([[0.4934, 0.5066]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
./ig_baselines/distilbert/1_1_7

### 4.1.5. Trying with random tokens

In [25]:
print("Generating random token sequences...")

for n in tqdm(range(1, 10 + 1)):
    for _ in tqdm(range(5000)):
        rand_token_seq = {
            "input_ids": torch.randint(0, len(tokenizer.vocab.items()), (1, n)).to(device), # 0-30521
            "attention_mask": torch.ones((1, n)).to(device)
        }

        # For generation with CLS and SEP tokens
        # rand_token_seq = {
        #     "input_ids": torch.randint(0, len(tokenizer.vocab.items()), (1, n + 2)).to(device), # 0-30521
        #     "attention_mask": torch.ones((1, n + 2)).to(device)
        # }

        # rand_token_seq["input_ids"][0,0] = 101
        # rand_token_seq["input_ids"][0,-1] = 102
        
        results = torch.softmax(model(**rand_token_seq).logits, dim=1)
        v1 = results[0, 0].item()
        v2 = results[0, 1].item()
        
        if (0.49 <= v1 <= 0.51):
            
            decoded_tokens = []
            for token in rand_token_seq["input_ids"][0]:
                decoded_tokens.append(tokenizer.decode(token, skip_special_tokens=False))
            sequence = " ".join(decoded_tokens)
            
            folder = Path(f"./ig_baselines/random_tokens/clean/distilbert/1_{n}_768/")
            # For generation with CLS and SEP tokens
            # folder = Path(f"./ig_baselines/random_tokens/firstlast/distilbert/1_{n}_768/")
            folder = folder / Path(f"{str(v1).replace(".", "")[:5]}_{str(v2).replace(".", "")[:5]}/")
            folder.mkdir(parents=True, exist_ok=True)
            torch.save(rand_token_seq["input_ids"], folder / Path("input_ids.pt"))
            torch.save(rand_token_seq["attention_mask"], folder / Path("attention_mask.pt"))
            (folder / Path("sequence.txt")).write_text(sequence, encoding="utf-8")

Generating random token sequences...


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

...

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

## 4.2. Searching a neutral output (Roberta)

In [12]:
# Aux function for this experiment
def forward_fn_rob(embeds):
    outputs = rob_model(inputs_embeds=embeds)
    return torch.softmax(outputs.logits, dim=1)

In [443]:
# Current baseline I/O
pad_emb = rob_model.get_input_embeddings().weight[rob_tokenizer.pad_token_id]
pad_emb = pad_emb.unsqueeze(0).unsqueeze(1)

print(pad_emb.shape, "-", forward_fn_rob(pad_emb)) # bias towards first class

# Replicating sequence len of the original example
pad_seq = pad_emb.repeat(1, 7, 1)
print(pad_seq.shape, "-", forward_fn_rob(pad_seq)) # this shows bias towards the second class for longer sequences

torch.Size([1, 1, 1024]) - tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 1024]) - tensor([[0.0179, 0.9821]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [450]:
# Current baseline for pipeline
Baseline = "" + rob_tokenizer.pad_token * 7

bl_tokens = rob_tokenizer(Baseline, return_tensors="pt").to(device)

# Checks if pipeline returns the same results as manual processing
print("Sequence:\n\t", Baseline, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Baseline), "==", torch.softmax(rob_model(**bl_tokens).logits, dim=1))
print()

bl_tokens = {
    "input_ids": bl_tokens.input_ids[:, 1:-1],
    "attention_mask": bl_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**bl_tokens).logits, dim=1))

Sequence:
	 <pad><pad><pad><pad><pad><pad><pad> 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.9070945382118225}] == tensor([[0.0929, 0.9071]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.2.1. Trying different textual inputs

#### Mask

In [451]:
Neutral_example = "" + rob_tokenizer.mask_token * 7 # Close to 0.5

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 <mask><mask><mask><mask><mask><mask><mask> 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9873067140579224}] == tensor([[0.9873, 0.0127]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.0147, 0.9853]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Sep

In [452]:
Neutral_example = "" + rob_tokenizer.sep_token * 7 # Close to 0.5

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 </s></s></s></s></s></s></s> 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.9350505471229553}] == tensor([[0.0649, 0.9351]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Cls

In [453]:
Neutral_example = "" + rob_tokenizer.cls_token * 7 # Close to 0.5

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 <s><s><s><s><s><s><s> 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9963724613189697}] == tensor([[0.9964, 0.0036]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Unk

In [454]:
Neutral_example = "" + rob_tokenizer.unk_token * 7 # Close to 0.5

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 <unk><unk><unk><unk><unk><unk><unk> 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9784794449806213}] == tensor([[0.9785, 0.0215]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.1277, 0.8723]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Average

In [455]:
Neutral_example = "average" + " average" * 6 # Close to 0.5

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 average average average average average average average 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9904770851135254}] == tensor([[0.9905, 0.0095]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.0163, 0.9837]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Dots

In [456]:
Neutral_example = "." + " ." * 6

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 . . . . . . . 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.513382613658905}] == tensor([[0.4866, 0.5134]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.0352, 0.9648]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [457]:
Neutral_example = "." * 7

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 ....... 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.9301257729530334}] == tensor([[0.0699, 0.9301]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Hyphen

In [458]:
Neutral_example = "-" + " -" * 6

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 - - - - - - - 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9725606441497803}] == tensor([[0.9726, 0.0274]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.8758, 0.1242]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [459]:
Neutral_example = "-" * 7

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 ------- 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9982836246490479}] == tensor([[0.9983, 0.0017]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Numbers

In [460]:
Neutral_example = "0" * 7

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 0000000 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.9362399578094482}] == tensor([[0.0638, 0.9362]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [461]:
Neutral_example = "0" + " 0" * 6

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 0 0 0 0 0 0 0 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9984257221221924}] == tensor([[0.9984, 0.0016]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9975, 0.0025]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [462]:
from random import randint

Neutral_example = "".join([str(randint(0, 9)) for n in range(7)])

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 7144360 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9672926068305969}] == tensor([[0.9673, 0.0327]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9728, 0.0272]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [463]:
from random import randint

Neutral_example = " ".join([str(randint(0, 9)) for n in range(7)])

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

Sequence:
	 2 0 3 0 8 1 0 

Results with first <s> and last </s> tokens:
	 [{'label': 'NEGATIVE', 'score': 0.9932148456573486}] == tensor([[0.9932, 0.0068]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.9903, 0.0097]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


#### Lorem ipsum

In [464]:
Neutral_example = "Lorem ipsum dolor sit amet consectetur adipiscing"

neutral_tokens = rob_tokenizer(Neutral_example, return_tensors="pt").to(device)
print(neutral_tokens.input_ids.shape)

print("Sequence:\n\t", Neutral_example, "\n")
print("Results with first <s> and last </s> tokens:\n\t", rob_classifier(Neutral_example), "==", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))
print()

neutral_tokens = {
    "input_ids": neutral_tokens.input_ids[:, 1:-1],
    "attention_mask": neutral_tokens.attention_mask[:, 1:-1]
}

print("Removing first <s> and last </s> tokens:\n\t", torch.softmax(rob_model(**neutral_tokens).logits, dim=1))

torch.Size([1, 19])
Sequence:
	 Lorem ipsum dolor sit amet consectetur adipiscing 

Results with first <s> and last </s> tokens:
	 [{'label': 'POSITIVE', 'score': 0.987579882144928}] == tensor([[0.0124, 0.9876]], device='cuda:3', grad_fn=<SoftmaxBackward0>)

Removing first <s> and last </s> tokens:
	 tensor([[0.0088, 0.9912]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.2.2. Creating a mean embedding

In [465]:
# Trying with a mean embedding...
emb_layer = rob_model.get_input_embeddings().weight # torch.Size([50265, 1024])
mean_emb = emb_layer.mean(dim=0, keepdim=True)      # torch.Size([1, 1024])
mean_emb = mean_emb.unsqueeze(1)                    # torch.Size([1, 1, 1024])

print(forward_fn_rob(mean_emb))

tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [466]:
# Trying with a longer sequence...
for n in range(1, 10 + 1):
    mean_seq = mean_emb.repeat(1, n, 1) 
    print(mean_seq.shape, "-", forward_fn_rob(mean_seq)) # this shows bias towards the second class for longer sequences and this scenario

torch.Size([1, 1, 1024]) - tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 1024]) - tensor([[0.8913, 0.1087]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 1024]) - tensor([[0.0218, 0.9782]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 1024]) - tensor([[0.0138, 0.9862]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 1024]) - tensor([[0.0147, 0.9853]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 1024]) - tensor([[0.0133, 0.9867]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 1024]) - tensor([[0.0124, 0.9876]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 1024]) - tensor([[0.0113, 0.9887]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 1024]) - tensor([[0.0105, 0.9895]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 1024]) - tensor([[0.0107, 0.9893]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [467]:
# Removing unused tokens...
special_tokens_idx = rob_tokenizer.all_special_ids
unused_tokens_idx = [n for n in range(999) if n not in special_tokens_idx]

used_mask = torch.ones(emb_layer.size(0), dtype=torch.bool)
used_mask[unused_tokens_idx] = False

mean_used_emb = emb_layer[used_mask].mean(dim=0, keepdim=True)
mean_used_emb = mean_used_emb.unsqueeze(1)

# results
print(mean_used_emb.shape)
print(forward_fn_rob(mean_used_emb)) # more biased towards the first class when compared to the first experiment

torch.Size([1, 1, 1024])
tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [468]:
# Trying last experiment with a longer sequence...
for n in range(1, 10 + 1):
    mean_used_seq = mean_used_emb.repeat(1, n, 1) 
    print(mean_used_seq.shape, "-", forward_fn_rob(mean_used_seq)) # this shows bias towards the second class for longer sequences again

torch.Size([1, 1, 1024]) - tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 1024]) - tensor([[0.8907, 0.1093]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 1024]) - tensor([[0.0220, 0.9780]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 1024]) - tensor([[0.0138, 0.9862]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 1024]) - tensor([[0.0148, 0.9852]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 1024]) - tensor([[0.0128, 0.9872]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 1024]) - tensor([[0.0124, 0.9876]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 1024]) - tensor([[0.0109, 0.9891]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 1024]) - tensor([[0.0096, 0.9904]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 1024]) - tensor([[0.0102, 0.9898]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.2.3. Trying with 0 embedding

In [697]:
zero_emb = torch.zeros((1,1,1024)).to(device)

# results
print(zero_emb.shape)
print(forward_fn_rob(zero_emb)) # more biased towards the first class when compared to the first experiment

torch.Size([1, 1, 1024])
tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [698]:
# Trying last experiment with a longer sequence...
for n in range(1, 10 + 1):
    zero_seq = torch.zeros((1, n, 1)).to(device)
    print(zero_seq.shape, "-", forward_fn_rob(zero_seq)) # this shows bias towards the second class for longer sequences again

torch.Size([1, 1, 1]) - tensor([[0.9989, 0.0011]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 2, 1]) - tensor([[0.9066, 0.0934]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 3, 1]) - tensor([[0.0165, 0.9835]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 4, 1]) - tensor([[0.0183, 0.9817]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 5, 1]) - tensor([[0.0143, 0.9857]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 6, 1]) - tensor([[0.0103, 0.9897]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 7, 1]) - tensor([[0.0126, 0.9874]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 8, 1]) - tensor([[0.0262, 0.9738]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 9, 1]) - tensor([[0.0595, 0.9405]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
torch.Size([1, 10, 1]) - tensor([[0.9855, 0.0145]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


### 4.2.4. Trying with random embeddings

In [699]:
rand_emb = torch.randn((1, 7, 1024)).to(device)

# results
print(rand_emb.shape)
print(forward_fn_rob(rand_emb)) # more biased towards the second class when compared to the first experiment

torch.Size([1, 7, 1024])
tensor([[0.0088, 0.9912]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [779]:
for n in range(10000):
    for m in range(1, 10+1):
        rand_emb = torch.randn((1, m, 1024)).to(device)

        results = forward_fn_rob(rand_emb)
        v1 = results[0, 0].item()
        v2 = results[0, 1].item()
        
        if (0.48 <= v1 <= 0.52):
            folder = Path(f"./ig_baselines/random_embs/roberta/1_{m}_1024/")
            filename = f"{str(v1).replace(".", "")[:5]}_{str(v2).replace(".", "")[:5]}.pt"
            filename =  folder / Path(filename)
            filename.parent.mkdir(parents=True, exist_ok=True)
            torch.save(rand_emb, filename)
            print(filename, results)

ig_baselines/roberta/1_8_1024/04949_05050.pt tensor([[0.4949, 0.5051]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_9_1024/05158_04841.pt tensor([[0.5158, 0.4842]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_10_1024/05145_04854.pt tensor([[0.5145, 0.4855]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_8_1024/05004_04995.pt tensor([[0.5004, 0.4996]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_9_1024/05043_04956.pt tensor([[0.5044, 0.4956]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_10_1024/05106_04893.pt tensor([[0.5107, 0.4893]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_3_1024/04871_05128.pt tensor([[0.4871, 0.5129]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_10_1024/05180_04819.pt tensor([[0.5180, 0.4820]], device='cuda:3', grad_fn=<SoftmaxBackward0>)
ig_baselines/roberta/1_4_1024/05053_04946.pt tensor([[0.5054,

### 4.2.5. Trying with random tokens

In [27]:
print("Generating random token sequences...")

for n in tqdm(range(1, 10 + 1)):
    for _ in tqdm(range(5000)):
        # rand_token_seq = {
        #     "input_ids": torch.randint(0, len(rob_tokenizer.vocab.items()), (1, n)).to(device), # 0-30521
        #     "attention_mask": torch.ones((1, n)).to(device)
        # }

        # For generation with <s> and </s> tokens
        rand_token_seq = {
            "input_ids": torch.randint(0, len(rob_tokenizer.vocab.items()), (1, n + 2)).to(device), # 0-30521
            "attention_mask": torch.ones((1, n + 2)).to(device)
        }

        rand_token_seq["input_ids"][0,0] = 0
        rand_token_seq["input_ids"][0,-1] = 2
        
        results = torch.softmax(rob_model(**rand_token_seq).logits, dim=1)
        v1 = results[0, 0].item()
        v2 = results[0, 1].item()
        
        if (0.49 <= v1 <= 0.51):
            
            decoded_tokens = []
            for token in rand_token_seq["input_ids"][0]:
                decoded_tokens.append(rob_tokenizer.decode(token, skip_special_tokens=False))
            sequence = " ".join(decoded_tokens)
            
            # folder = Path(f"./ig_baselines/random_tokens/clean/roberta/1_{n}_1024/")
            # For generation with <s> and </s> tokens
            folder = Path(f"./ig_baselines/random_tokens/firstlast/roberta/1_{n}_1024/")
            folder = folder / Path(f"{str(v1).replace(".", "")[:5]}_{str(v2).replace(".", "")[:5]}/")
            folder.mkdir(parents=True, exist_ok=True)
            torch.save(rand_token_seq["input_ids"], folder / Path("input_ids.pt"))
            torch.save(rand_token_seq["attention_mask"], folder / Path("attention_mask.pt"))
            (folder / Path("sequence.txt")).write_text(sequence, encoding="utf-8")

Generating random token sequences...


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

...

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

# 5. Loading generated embeddings

In [35]:
# Loading a random embedding from dict
bl_emb_path: Path = Path("./ig_baselines/random_tokens/clean/distilbert/1_7_768/05017_04982")

baseline_tokens = {
    "input_ids": torch.load(bl_emb_path / Path("input_ids.pt")).to(device),
    "attention_mask": torch.load(bl_emb_path / Path("attention_mask.pt")).to(device)
}

print(" ".join(tokenizer.batch_decode(baseline_tokens["input_ids"], skip_special_tokens=False)))
print(torch.softmax(model(**baseline_tokens).logits, dim=1))

titanium 之 usherwisekong altering italianate
tensor([[0.5017, 0.4983]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


In [38]:
# Loading a random embedding
bl_emb_path_2: Path = Path("./ig_baselines/random_embs/roberta/1_7_1024/05026_04973.pt")

bl_emb_2 = torch.load(bl_emb_path_2).to(device)

print(forward_fn_rob(bl_emb_2))

tensor([[0.5027, 0.4973]], device='cuda:3', grad_fn=<SoftmaxBackward0>)


A set of reference embeddings for computing Integrated Gradients is provided in the `ig_baselines` directory. The generation process is organized into two experimental configurations:

**1. Direct generation of random embeddings (Section 4.X.4, `random_embs` directory)**
- Contains embeddings corresponding to sequences of 1 to 10 tokens.
- Directories are named `1_n_768` for DistilBERT or `1_n_1024` for RoBERTa, where `n` represents the number of tokens.
- Each directory includes the corresponding embedding file, named according to the prediction score obtained (e.g., `05026_04973.pt` indicates 0.5026 for the first class and 0.4973 for the second).

**2. Generation of embeddings from valid `input_ids` (Section 4.X.5, `random_tokens` directory)**
- This configuration is divided into two structural variants:
  - `clean`: sequences without start or end tokens.
  - `firstlast`: sequences delimited with `CLS`/`SEP` (DistilBERT) or `<s>`/`</s>` (RoBERTa), depending on the model architecture.
- The directory structure is identical to the first case once inside the `clean` or `firstlast` folders. The main difference lies in how the sequences are stored: here, `input_ids.pt` and `attention_mask.pt` are stored separately within the `xxxxx_yyyyy` folders (which are named according to the predicted scores).
- Additionally, the file `sequence.txt` is provided, containing the plain-text decoding of the original token sequence.

**Implementation Instructions**
- Load the embeddings following the structure and patterns shown in the previous cell.
- It is recommended to prioritize the embeddings derived from `input_ids` as the primary baseline. As an alternative, baselines constructed from special tokens can be used by selecting one of the scenarios documented in Section 4.X.1.